What Stage 4 does and why it's structured this waySIMP (Solid Isotropic Material with Penalization) is the topology optimization algorithm. It starts with a uniform density field across all elements — every element is 50% material — and iteratively updates it by solving the FEA problem, computing sensitivities (how much does compliance change if I remove a tiny bit of material here?), and shifting density toward 0 (void) or 1 (solid) based on those sensitivities.The key insight is the penalization: density is raised to a power p (typically 3) before being multiplied into the stiffness. This makes intermediate densities mechanically inefficient — an element at 0.5 density has only 0.5³ = 0.125 of its full stiffness, so the optimizer is pushed toward binary 0/1 solutions rather than a grey mush.The data flow for this stage is:outputs/meshes/<name>_stage03.json       ← handoff from Stage 3
outputs/meshes/<name>.xdmf              ← mesh (re-used for FEA iterations)
outputs/meshes/<name>_boundaries.xdmf   ← BCs (re-used each iteration)
        ↓
src/optimization/density_filter.py       ← smooths density to avoid checkerboard
src/optimization/simp.py                 ← optimization loop
        ↓
outputs/meshes/<name>_density.xdmf       ← final density field
outputs/reports/<name>_density_*.png     ← renders at each checkpoint
outputs/reports/<name>_convergence.png   ← compliance history plot
outputs/meshes/<name>_stage04.json       ← handoff to Stage 5The split between density_filter.py and simp.py is load-bearing — without the filter, SIMP produces checkerboard instability (alternating solid/void elements in a checkerboard pattern that has artificially high stiffness but is physically unmakeable). The filter is what makes the result printable.

Cell 0 — Parameters (tag: parameters)

In [ ]:
# Cell 0 — tagged: parameters
import sys
sys.path.insert(0, "/workspace")

STAGE03_HANDOFF   = None    # auto-detect if None
VOLUME_FRACTION   = 0.4     # retain 40% of original material
PENAL             = 3.0     # penalization exponent — don't go below 2.5
FILTER_RADIUS     = 6.0     # mm — should be ~3x target_element_size
MAX_ITERATIONS    = 100
CONVERGENCE_TOL   = 1e-3
MOVE_LIMIT        = 0.2     # OC max density change per iteration
SAFETY_FACTOR_MIN = 1.2     # abort optimization if SF drops below this
CHECKPOINT_EVERY  = 10      # render + save density every N iterations

Cell 1 — Load Stage 3 handoff

In [ ]:
# Cell 1 — Read handoff from 03_fea_fenicsx.ipynb
import json
from pathlib import Path

if STAGE03_HANDOFF is None:
    candidates = sorted(Path("outputs/meshes").glob("*_stage03.json"))
    if not candidates:
        raise FileNotFoundError(
            "No stage03 handoff in outputs/meshes/. "
            "Run 03_fea_fenicsx.ipynb first."
        )
    handoff_path = candidates[-1]
else:
    handoff_path = Path(STAGE03_HANDOFF)

handoff = json.loads(handoff_path.read_text())
print(f"Loaded handoff: {handoff_path}")
print(json.dumps(handoff, indent=2))

part_name       = handoff["part_name"]
xdmf_path       = Path(handoff["xdmf_path"])
boundaries_xdmf = Path(handoff["xdmf_boundaries"])
load_hints      = handoff["load_hints"]
material        = handoff["material"]
baseline_sf     = handoff["safety_factor"]

print(f"\nBaseline safety factor: {baseline_sf:.2f}")
if baseline_sf < SAFETY_FACTOR_MIN:
    raise ValueError(
        f"Baseline safety factor {baseline_sf:.2f} is already below "
        f"SAFETY_FACTOR_MIN={SAFETY_FACTOR_MIN}. "
        f"Fix geometry or loads in Stage 1/3 before optimizing."
    )

Cell 2 — Configure SIMP

In [ ]:
# Cell 2 — Build SIMP config — inspectable before the expensive loop
from src.optimization.simp import SIMPConfig

config = SIMPConfig(
    volume_fraction  = VOLUME_FRACTION,
    penal            = PENAL,
    filter_radius    = FILTER_RADIUS,
    max_iterations   = MAX_ITERATIONS,
    convergence_tol  = CONVERGENCE_TOL,
    move             = MOVE_LIMIT,
    safety_factor_min= SAFETY_FACTOR_MIN,
    checkpoint_every = CHECKPOINT_EVERY,
)

print("SIMP configuration:")
print(f"  Volume fraction:  {config.volume_fraction} "
      f"(retaining {config.volume_fraction*100:.0f}% of material)")
print(f"  Penalization:     {config.penal}")
print(f"  Filter radius:    {config.filter_radius} mm")
print(f"  Max iterations:   {config.max_iterations}")
print(f"  Convergence tol:  {config.convergence_tol}")
print(f"  Move limit:       {config.move}")
print(f"  Min safety factor:{config.safety_factor_min}")

Cell 3 — Run optimization

In [ ]:
# Cell 3 — Run SIMP loop
# This is the long-running cell. At 100 iterations on a ~50k element mesh
# expect 20-60 minutes depending on solver. Use MUMPS for meshes < 500k DOFs.
import pyvista as pv
from pathlib import Path

pv.OFF_SCREEN = True
report_dir = Path("outputs/reports")

# Checkpoint callback — renders density field every CHECKPOINT_EVERY iterations
def checkpoint_render(iteration: int, rho: np.ndarray, compliance: float):
    from src.optimization.simp import W  # reuse the DG0 space
    # Write a quick PNG snapshot of the current density field
    density_mesh = pv.read(str(Path("outputs/meshes") / f"{part_name}_density.xdmf"))
    pl = pv.Plotter()
    pl.add_mesh(
        density_mesh,
        scalars="density",
        cmap="bone",
        clim=[0, 1],
        show_edges=False,
        scalar_bar_args={"title": "Density"},
    )
    pl.add_axes()
    pl.view_isometric()
    png = report_dir / f"{part_name}_density_iter{iteration:04d}.png"
    pl.screenshot(str(png))
    pl.close()
    print(f"    Checkpoint render: {png}")

from src.optimization.simp import run_simp

result = run_simp(
    xdmf_path=xdmf_path,
    boundaries_xdmf=boundaries_xdmf,
    part_name=part_name,
    output_dir="outputs/meshes",
    load_hints=load_hints,
    material=material,
    config=config,
    checkpoint_callback=checkpoint_render,
)

print(f"\nSuccess:          {result.success}")
print(f"Converged:        {result.converged}")
print(f"Iterations:       {result.n_iterations}")
print(f"Final compliance: {result.final_compliance:.4e}")
print(f"Final vol frac:   {result.final_volume_frac:.3f}")
print(f"Duration:         {result.duration_s}s "
      f"({result.duration_s/60:.1f} min)")

result.raise_if_failed()

Cell 4 — Convergence plots

In [ ]:
# Cell 4 — Plot compliance and volume fraction history
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

iters = np.arange(1, len(result.compliance_history) + 1)

ax1.semilogy(iters, result.compliance_history, color="steelblue", linewidth=1.5)
ax1.set_ylabel("Compliance (log scale)")
ax1.set_title(f"{part_name} — SIMP convergence")
ax1.axvline(result.n_iterations, color="gray", linestyle="--", alpha=0.5)
ax1.grid(True, alpha=0.3)

ax2.plot(iters, result.volume_history, color="coral", linewidth=1.5)
ax2.axhline(config.volume_fraction, color="gray", linestyle="--",
            label=f"target vf={config.volume_fraction}")
ax2.set_ylabel("Volume fraction")
ax2.set_xlabel("Iteration")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
conv_path = report_dir / f"{part_name}_convergence.png"
plt.savefig(conv_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Convergence plot: {conv_path}")

Cell 5 — Final density render

In [ ]:
# Cell 5 — Render final density field with threshold isosurface
# The 0.5 threshold is the standard cutoff: elements with rho > 0.5 are "solid"
# This is what Stage 5 will threshold and convert to STL

import pyvista as pv
from pathlib import Path

pv.OFF_SCREEN = True

density_mesh = pv.read(str(result.density_path))

# Full density field
pl = pv.Plotter(shape=(1, 2))

pl.subplot(0, 0)
pl.add_mesh(density_mesh, scalars="density", cmap="bone",
            clim=[0, 1], show_edges=False)
pl.add_text("Full density field", font_size=10)
pl.view_isometric()

# Thresholded — solid elements only (rho > 0.5)
solid = density_mesh.threshold(0.5, scalars="density")
pl.subplot(0, 1)
pl.add_mesh(solid, color="lightsteelblue", show_edges=True)
pl.add_text("Thresholded (ρ > 0.5)", font_size=10)
pl.view_isometric()

final_png = report_dir / f"{part_name}_density_final.png"
pl.screenshot(str(final_png))
pl.close()
print(f"Final density render: {final_png}")

solid_fraction = solid.n_cells / density_mesh.n_cells
print(f"Solid elements (ρ>0.5): {solid.n_cells:,} / "
      f"{density_mesh.n_cells:,} ({solid_fraction*100:.1f}%)")

Cell 6 — Write Stage 4 handoff

In [ ]:
# Cell 6 — Handoff for 05_stl_export.ipynb
import json
from pathlib import Path

handoff_out = {
    "stage":              "04_simp",
    "part_name":          part_name,
    "density_path":       str(result.density_path),
    "xdmf_path":          str(xdmf_path),
    "n_iterations":       result.n_iterations,
    "converged":          result.converged,
    "final_compliance":   result.final_compliance,
    "final_volume_frac":  result.final_volume_frac,
    "duration_s":         result.duration_s,
    "config": {
        "volume_fraction": config.volume_fraction,
        "penal":           config.penal,
        "filter_radius":   config.filter_radius,
        "convergence_tol": config.convergence_tol,
    },
    "material":           material,
    "load_hints":         load_hints,
}

handoff_path_out = Path("outputs/meshes") / f"{part_name}_stage04.json"
handoff_path_out.write_text(json.dumps(handoff_out, indent=2))
print(f"Handoff written: {handoff_path_out}")
print(json.dumps(handoff_out, indent=2))

How Stage 4 connects to Stage 5
Two things travel forward:
density_path — the XDMF file containing the final rho field on the DG0 mesh. Stage 5 reads this, applies a threshold (default 0.5), and runs marching cubes via scikit-image to extract the isosurface as a triangle mesh. The threshold is configurable in Stage 5 — raising it (e.g. 0.6) gives a more conservative, heavier result; lowering it (e.g. 0.4) gives a lighter but more fragile one.
xdmf_path — the original mesh. Stage 5 needs the node coordinates to map the DG0 density field onto a regular voxel grid before marching cubes can run, since marching cubes expects a structured grid rather than an unstructured tet mesh.
The most important parameter connecting Stage 4 to Stage 5 is filter_radius. If it's too small relative to target_element_size, the density field will contain checkerboard noise that marching cubes turns into physically unmakeable thin spikes and disconnected islands in the STL. If your Stage 5 STL looks spiky or has disconnected components, come back here and increase FILTER_RADIUS by 50% and re-run from Cell 3.